In [1]:
from pathlib import Path
import pandas as pd

ALLOWED_LABELS = ['Happy', 'Sad', 'Angry', 'Energetic']
MAX_SONGS_PER_LABEL = 5000

songs_ready_path = Path('/kaggle/input/datasets/ishita212005/is450tm1/songs_ready_for_classifier.csv')
if not songs_ready_path.exists():
    raise FileNotFoundError('Run the feature matrix section above first so songs_ready_for_classifier.csv exists.')

songs_transformer = pd.read_csv(songs_ready_path)
llm_path = Path('/kaggle/input/datasets/ishita212005/is450tm/llm_annotated_8500.csv')

if llm_path.exists():
    transformer_labels = pd.read_csv(llm_path)
    transformer_labels = transformer_labels[transformer_labels['emotion_label'].isin(ALLOWED_LABELS)][['id', 'emotion_label']].copy()
    label_source = '/kaggle/input/datasets/ishita212005/is450tm/llm_annotated_8500.csv'
else:
    cluster_path = Path('songs_with_clusters.csv')
    if not cluster_path.exists():
        raise FileNotFoundError('Run the clustering section first so songs_with_clusters.csv exists.')
    cluster_df = pd.read_csv(cluster_path, usecols=['id', 'emotion_cluster'])
    transformer_labels = cluster_df.rename(columns={'emotion_cluster': 'emotion_label'})
    transformer_labels = transformer_labels[transformer_labels['emotion_label'].isin(ALLOWED_LABELS)].copy()
    label_source = 'songs_with_clusters.csv (weak labels from KMeans emotion clusters)'

transformer_df = songs_transformer[['id', 'lyrics_normalized']].merge(
    transformer_labels,
    on='id',
    how='inner'
).dropna().copy()
transformer_df['lyrics_normalized'] = transformer_df['lyrics_normalized'].astype(str)
transformer_df = transformer_df[transformer_df['lyrics_normalized'].str.strip().ne('')].reset_index(drop=True)
transformer_df = (
    transformer_df.groupby('emotion_label', group_keys=False)
    .apply(lambda df: df.sample(n=min(MAX_SONGS_PER_LABEL, len(df)), random_state=42))
    .reset_index(drop=True)
)

print('Label source:', label_source)
print(transformer_df['emotion_label'].value_counts())
transformer_df.to_csv('transformer_training_data.csv', index=False)
print('Saved transformer_training_data.csv')

/tmp/ipykernel_55/3521631385.py:36: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda df: df.sample(n=min(MAX_SONGS_PER_LABEL, len(df)), random_state=42))


Label source: /kaggle/input/datasets/ishita212005/is450tm/llm_annotated_8500.csv
emotion_label
Sad          4044
Energetic    3330
Angry        2404
Happy        1499
Name: count, dtype: int64
Saved transformer_training_data.csv


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

WORDS_PER_CHUNK = 180
CHUNK_STRIDE = 120
MIN_CHUNK_WORDS = 40

def chunk_lyrics(text, words_per_chunk=WORDS_PER_CHUNK, stride=CHUNK_STRIDE):
    words = str(text).split()
    if not words:
        return []
    chunks = []
    start = 0
    while start < len(words):
        chunk_words = words[start:start + words_per_chunk]
        if len(chunk_words) < MIN_CHUNK_WORDS and chunks:
            break
        chunks.append(' '.join(chunk_words))
        if start + words_per_chunk >= len(words):
            break
        start += stride
    return chunks or [' '.join(words[:words_per_chunk])]

song_level_df = transformer_df[['id', 'lyrics_normalized', 'emotion_label']].drop_duplicates().reset_index(drop=True)
chunk_rows = []
for row in song_level_df.itertuples(index=False):
    for chunk_id, chunk_text in enumerate(chunk_lyrics(row.lyrics_normalized)):
        chunk_rows.append({
            'id': row.id,
            'chunk_id': chunk_id,
            'text': chunk_text,
            'emotion_label': row.emotion_label,
        })
chunk_df = pd.DataFrame(chunk_rows)

train_songs, temp_songs = train_test_split(
    song_level_df[['id', 'emotion_label']],
    test_size=0.2,
    random_state=42,
    stratify=song_level_df['emotion_label'],
)
val_songs, test_songs = train_test_split(
    temp_songs,
    test_size=0.5,
    random_state=42,
    stratify=temp_songs['emotion_label'],
)

train_chunks = chunk_df[chunk_df['id'].isin(train_songs['id'])].reset_index(drop=True)
val_chunks = chunk_df[chunk_df['id'].isin(val_songs['id'])].reset_index(drop=True)
test_chunks = chunk_df[chunk_df['id'].isin(test_songs['id'])].reset_index(drop=True)

print('Song-level split sizes:')
print('Train songs:', len(train_songs))
print('Val songs:  ', len(val_songs))
print('Test songs: ', len(test_songs))
print('\nChunk-level split sizes:')
print('Train chunks:', len(train_chunks))
print('Val chunks:  ', len(val_chunks))
print('Test chunks:', len(test_chunks))

Song-level split sizes:
Train songs: 9021
Val songs:   1128
Test songs:  1128

Chunk-level split sizes:
Train chunks: 21038
Val chunks:   2673
Test chunks: 2597


In [3]:
import numpy as np
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

MODEL_NAME = 'j-hartmann/emotion-english-distilroberta-base'
MAX_LENGTH = 256
label_list = [label for label in ALLOWED_LABELS if label in sorted(transformer_df['emotion_label'].unique())]
label2id = {label: idx for idx, label in enumerate(label_list)}
id2label = {idx: label for label, idx in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

def make_dataset(frame):
    dataset = Dataset.from_pandas(frame[['text', 'emotion_label']].copy(), preserve_index=False)
    dataset = dataset.map(
        lambda batch: tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH),
        batched=True,
    )
    dataset = dataset.map(
        lambda batch: {'labels': [label2id[label] for label in batch['emotion_label']]},
        batched=True,
    )
    return dataset.remove_columns(['text', 'emotion_label'])
train_dataset = make_dataset(train_chunks)
val_dataset = make_dataset(val_chunks)
test_dataset = make_dataset(test_chunks)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average='macro'),
        'weighted_f1': f1_score(labels, preds, average='weighted'),
    }

training_args = TrainingArguments(
    output_dir='transformer_emotion_model',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    report_to='none',
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model('transformer_emotion_model')
tokenizer.save_pretrained('transformer_emotion_model')

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('transformer_emotion_model/tokenizer_config.json',
 'transformer_emotion_model/tokenizer.json')

In [4]:
from sklearn.metrics import classification_report

prediction_output = trainer.predict(test_dataset)
chunk_probabilities = torch.softmax(torch.tensor(prediction_output.predictions), dim=1).numpy()
probability_columns = [f'prob_{id2label[idx]}' for idx in range(len(id2label))]

chunk_eval_df = test_chunks[['id', 'chunk_id', 'emotion_label']].copy()
chunk_eval_df[probability_columns] = chunk_probabilities
song_level_predictions = chunk_eval_df.groupby(['id', 'emotion_label'])[probability_columns].mean().reset_index()
song_level_predictions['predicted_label'] = song_level_predictions[probability_columns].idxmax(axis=1).str.replace('prob_', '', regex=False)

print(classification_report(
    song_level_predictions['emotion_label'],
    song_level_predictions['predicted_label'],
    labels=label_list,
))

song_level_predictions.to_csv('transformer_song_level_predictions.csv', index=False)
print('Saved transformer_song_level_predictions.csv')
song_level_predictions[['emotion_label', 'predicted_label']].to_csv('bert_predictions.csv', index=False)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


              precision    recall  f1-score   support

       Happy       0.84      0.57      0.68       150
         Sad       0.74      0.91      0.81       404
       Angry       0.64      0.53      0.58       241
   Energetic       0.74      0.73      0.73       333

    accuracy                           0.73      1128
   macro avg       0.74      0.68      0.70      1128
weighted avg       0.73      0.73      0.72      1128

Saved transformer_song_level_predictions.csv


In [5]:
songs_df      = pd.read_csv('/kaggle/input/datasets/ishita212005/is450tm1/songs_ready_for_classifier.csv')
training_data = pd.read_csv('/kaggle/working/transformer_training_data.csv')
seen_ids      = set(training_data['id'].unique())
unseen_df     = songs_df[~songs_df['id'].isin(seen_ids)].reset_index(drop=True)



In [6]:
demo_song = unseen_df.sample(n=1, random_state=30).iloc[0]
pd.DataFrame([demo_song]).to_csv('demo_song.csv', index=False)
print(f"Saved: {demo_song['name']} by {demo_song['artists']}")

Saved: Love To The Sound Emergency by ["Red Knife Lottery"]


In [7]:
import pandas as pd, numpy as np, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained('transformer_emotion_model')
model     = AutoModelForSequenceClassification.from_pretrained('transformer_emotion_model')
model.eval()
id2label  = model.config.id2label

demo_song = pd.read_csv('demo_song.csv').iloc[0]
print(f"🎵 {demo_song['name']} — {demo_song['artists']} ({demo_song['genre']})")
print(f"\n {demo_song['lyrics_normalized'][:300]}...")

chunks    = chunk_lyrics(demo_song['lyrics_normalized'])
all_probs = []
for chunk in chunks:
    inputs = tokenizer(chunk, return_tensors='pt', truncation=True, max_length=256)
    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits, dim=1).squeeze().numpy()
    all_probs.append(probs)

avg_probs = np.mean(all_probs, axis=0)
predicted = id2label[int(np.argmax(avg_probs))]

for j, label in id2label.items():
    print(f"  {label:<12}: {avg_probs[j]:.3f}  {'█' * int(avg_probs[j]*30)}")
print(f"\n {predicted}  ({avg_probs[int(np.argmax(avg_probs))]:.1%} confidence)")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

🎵 Love To The Sound Emergency — ["Red Knife Lottery"] (Rock)

 we need stimulation. musical masturbation. what better way to satisfy than let the guitars gratify. pound this sound. pump pump the heart the blood this love. love to the sound emergency. we sing medication in this khaki prozac nation. no need to postulate. just let this love reverberate. dance this...
  Happy       : 0.011  
  Sad         : 0.176  █████
  Angry       : 0.100  ███
  Energetic   : 0.713  █████████████████████

 Energetic  (71.3% confidence)
